# 10. LightGBM: внешние факторы и соседи по графам (Kaggle)

Код берётся из GitHub (`Kinggl3/vkr-transit`), данные — датасет `vkr-transit-data`.

Расчёт разбит на **две части, которые запускаются параллельно** (каждая укладывается в лимит Kaggle 12 ч):

| часть | модели | точки прогноза | время |
|---|---|---|---|
| **A** | `lgbm` (без соседей, H1), `lgbm_all` (все графы, H2) | все точки val и test | ~4–5 ч |
| **B** | `lgbm_line`, `lgbm_dist`, `lgbm_od_in`, `lgbm_od_dyn` (H3), `lgbm_wactual` (H5) | общий подсет (каждая 7-я) | ~4 ч |

**Как запустить:** импортировать ноутбук дважды; в первой ячейке одного оставить `PART = "A"`, в другом поставить `"B"`.
Настройки: Accelerator — None (CPU), Internet — On; Input — датасет `vkr-transit-data`. Save & Run All (Commit).
Оценка всех моделей вместе — ноутбук 11 (подключает вывод обеих частей).

In [ ]:
PART = "A"   # "A" или "B"

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## Модели части

In [ ]:
RUNS = {   # имя: (графы соседей, погода на val/test, выборки, шаг точек прогноза: 1 — все, 7 — общий подсет)
    "A": [("lgbm",         [],                                  "forecast", "val test", 1),
          ("lgbm_all",     ["line", "dist", "od_in", "od_dyn"], "forecast", "val test", 1)],
    "B": [("lgbm_line",    ["line"],                            "forecast", "val test", 7),
          ("lgbm_dist",    ["dist"],                            "forecast", "val test", 7),
          ("lgbm_od_in",   ["od_in"],                           "forecast", "val test", 7),
          ("lgbm_od_dyn",  ["od_dyn"],                          "forecast", "val test", 7),
          ("lgbm_wactual", [],                                  "actual",   "val",      7)],
}[PART]
for name, graphs, weather, splits, ps in RUNS:
    need = [s for s in splits.split() if not os.path.exists(f"{R}/preds/{name}__{s}.npz")]
    if not need:
        print(f"{name}: уже посчитан"); continue
    t = time.time()
    g = f"--graphs {' '.join(graphs)}" if graphs else ""
    sp = " ".join(need)
    !cd {REPO} && python src/train_lgbm.py --name {name} {g} --weather {weather} --splits {sp} --pred-stride {ps}
    print(f"=== {name}: {(time.time() - t) / 60:.0f} мин\n", flush=True)

## Итог части

In [ ]:
import pandas as pd
print(pd.read_csv(f"{R}/runs.csv")[["model", "split", "fit_s", "predict_s", "params"]].to_string())
print("Прогнозы:", sorted(os.path.basename(p) for p in glob.glob(f"{R}/preds/*.npz")))